# Практика · Морфологія: лематизація і стемінг> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) · Домашнє завдання: [homework.html](homework.html)⏱ **Заміряно: близько хвилини** на чотирьох ядрах без відеокарти.Найдовше працюють два місця: лематизація ста тисяч слововживань без кешуй перебір довжин списку стоп-слів.Тут ми **рахуємо своїм кодом** кожне число, яке лекція називає словами:1. збираємо корпус українських перекладів і токенізуємо його;2. проходимо сходинки нормалізації й дивимось, що дає кожна;3. лематизуємо словник і міряємо, на скільки він стискається;4. будуємо наївний стемер і міряємо **дві** його ціни одночасно;5. рахуємо омонімію і показуємо, де перший розбір хибний;6. будуємо задачу, де лематизація **допомагає**;7. будуємо задачу, де вона **шкодить** — на золотій розмітці з файлів перекладів;8. будуємо власний список стоп-слів і дивимось, скільки він коштує;9. міряємо час із кешем і без.

## 0 · СередовищеДрукуємо версії — щоб пізніше було зрозуміло, у чому саме рахувалось.

In [ ]:
import sys, re, glob, gettext, random, time
from collections import Counter, defaultdict

import numpy as np
import sklearn
import pymorphy3

print("Python      ", sys.version.split()[0])
print("NumPy       ", np.__version__)
print("scikit-learn", sklearn.__version__)
print("pymorphy3   ", pymorphy3.__version__)

## 1 · Корпус: українські переклади в системіТой самий корпус, що й у темах 01-02: файли `.mo` кожної встановленої програми,у яких лежать пари «англійський оригінал → український переклад».⚠️ **Без української локалі цей зошит до кінця не виконається.** Нижче стоїть запобіжник, який вмикає вбудований мінікорпус, і перші розділи на ньому справді працюють. Але це сотня документів замість сотень тисяч, і далі розділам просто бракує матеріалу: відсів лишає `CountVectorizer` без ознак, а вибірки просять більше прикладів, ніж є. Зошит упаде на зрозумілій помилці, а не мовчки — це заміряно, а не припущено. Перевірити, чи корпус є: `ls /usr/share/locale/uk/LC_MESSAGES/*.mo | wc -l` — потрібно приблизно від сотні файлів.

In [ ]:
MINI_CORPUS = """
no return statement in function returning non-void ||| в функції, що повертає не void, відсутня інструкція return
cannot acquire state change lock (held by monitor=%1$s) ||| не вдалося створити блокування зміни станів (утримується monitor=%1$s)
a connection using '%s' authentication cannot specify WPA protocols ||| з'єднання, де використовується розпізнавання «%s», не може використовувати специфічні протоколи WPA
Device %s is not a valid FVAULT2 device. ||| Пристрій %s не є коректним пристроєм FVAULT2.
No address associated with hostname ||| З цією назвою вузла не пов’язано жодної адреси
failed to lookup interface with MAC address '%1$s' ||| не вдалося знайти інтерфейс з MAC-адресою «%1$s»
specified text search configuration '%s' might not match locale '%s' ||| вказана конфігурація текстового пошуку '%s' може не підходити локалі '%s'
Not supported on this platform ||| На цій платформі підтримки не передбачено
unable to get server IP addr ||| не вдалося отримати IP-адресу сервера
Cannot specify storage and use --nodisks ||| Не можна одночасно вказувати сховище даних і використовувати --nodisks
invalid privilege type %s for column ||| недійсний тип права %s для стовпця
cannot create a temporary relation as partition of permanent relation '%s' ||| створити тимчасове відношення як секцію постійного відношення'%s' не можна
No database connection exists to re-use parameters from ||| Не існує підключення до бази даних для повторного використання параметрів
could not look up effective user ID %ld: %s ||| не можу знайти користувача з ефективним ID %ld: %s
including column does not support an operator class ||| включені стовпці не підтримують класи операторів
%qT is not constructible from %qE ||| %qT не може бути сконструйованим з %qE
this msdos-style partition label has no post-MBR gap; embedding won't be possible ||| ця мітка розділу у форматі msdos не містить проміжку після MBR, — вбудовування неможливе
selected architecture does not support wide conditional branch instruction ||| у вибраній архітектурі не передбачено підтримки інструкції широкого умовного розгалуження
%<-fmoduleinfo%> is not supported on this target ||| %<-fmoduleinfo%> не підтримується на цій цілі
Not saving repeating crash after %ds (limit is %ds) ||| Не збережено дані повторного аварійного завершення за %d с (обмеження — %d с)
invalid %%y value, try using the 'Z' constraint ||| недійсне значення %%y, спробуйте використати обмеження 'Z'
subscription has no replication slot set ||| для підписки не встановлений слот реплікації
not compressing section data: zlib error ||| дані розділу не буде стиснуто: помилка zlib
range %1$s - %2$s is not entirely within network %3$s/%4$d ||| діапазон %1$s - %2$s не повністю лежить у мережі %3$s/%4$d
Can't continue without Red Hat Support case number ||| Продовження неможливе без номера справи Red Hat Support
Could not load IPv6 user interface. ||| Не вдалось завантажити інтерфейс користувача налаштовування IPv6
Variable %qs at %L may not be a C interoperable kind but it is BIND(C) ||| Змінна %qs на %L може не бути сумісним з C видом, але має атрибут BIND(C)
tex4ht.pm: closing communication failed: %s: %s ||| tex4ht.pm: не вдалося завершити обмін даними: %s: %s
could not close large object TOC file '%s': %m ||| не вдалося закрити великий об'єкт файлу TOC '%s' %m
invalid address for 'S' output modifier ||| неправильна адреса для модифікатора виводу 'S'
--build-id argument '%s' not a valid hex number ||| аргумент --build-id, «%s», не є коректним шістнадцятковим числом
Invalid configuration of CCpp addon, unsupported Package manager: '%s' ||| Некоректне налаштування додатка CCpp, непідтримуваний засіб керування пакунками: «%s»
target system does not support debug output ||| система ціль не підтримує вивід налагодження
Support for writing AIX disk labels is is not implemented yet. ||| Підтримка запису позначок дисків у стилі AIX ще не реалізована.
No valid or appropriate certificate for “%s” was found ||| Не знайдено дійсного або відповідного сертифіката для «%s»
selected processor does not support PACBTI extention ||| у вибраному процесорі не передбачено підтримки розширення PACBTI
No block for an inode with inline data ||| Немає блоку для inode із вбудованими даними
Could not write to port (%s) ||| Не вдалося записати у порт (%s)
invalid use of member %qD (did you forget the %<&%> ?) ||| неправильне використання члена %qD (чи ви забули %<&%> ?)
Cannot record working tree state ||| Неможливо записати стан робочого дерева
stack limits not supported on this target ||| обмеження стеку не підтримуються на даній платформі
cannot read program interpreter ||| не вдалося прочитати інтерпретатор програми
do not set new values, but only display the current ones. ||| не встановлювати нові значення, а лише показати поточні.
machine paused, so can't power it down ||| роботу машини призупинено, отже її не можна вимикати
The OCSP response's signature cannot be validated. ||| Підпис відповіді OCSP не пройшов перевірки.
The program is not marked as executable. ||| Цю програму не позначено як виконувану.
structure alignment must be a small power of two, not %wu ||| вирівнювання структури повинно бути невеликою степенню двійки, а не %wu
Failed to delete all child files ||| Не вдалося вилучити усі дочірні файли
rule '%s' for relation '%s' does not exist ||| правило '%s' для відношення '%s' не існує
Failed to count network filters ||| Не вдалося полічити фільтри мережі
invalid encoding prefix in literal operator ||| недійсний префікс кодування в літеральному операторі
Removable media not supported for %1$s device ||| Для пристрою %1$s не передбачено підтримки портативних носіїв даних
%<critical%> region may not be nested inside a %<critical%> region with the same name ||| Регіон %<critical%> не може бути вкладений у регіон %<critical%> з таким самим імʼям
no unique final overrider for %qD in %qT ||| немає єдиного остаточного перекривача для %qD в %qT
Graphical installation is not available. Starting text mode. ||| Графічне встановлення неможливе. Запускається текстовий режим.
Image type is not supported ||| Зображення цього типу не підтримуються
Remote OCI index has no registry uri ||| У покажчику віддаленого OCI немає адреси реєстру
%s: could not locate matching postgres executable ||| %s: не вдалося знайти відповідний postgres файл, що виконується
Type %s does not implement from_tokens() on the GIcon interface ||| Для типу %s не реалізовано from_tokens() у інтерфейсі GIcon
called object is not a function or function pointer ||| обʼєкт, що був викликаний, не є функцією або вказівником на функцію
In the directive number %u, '~:[' is not followed by two clauses, separated by '~;'. ||| У директиві з номером %u, '~:[' не завершується двома реченнями, розділеними  '~;'.
'%s' is not a valid %s address for '%s' option ||| «%s» не є припустимою адресою %s для параметра «%s»
add_cacheinfo failed for path '%s'; merge aborting. ||| невдала спроба add_cacheinfo для шляху '%s'; переривання злиття.
$alias device ${DEVICE} does not seem to be present, delaying initialization. ||| Схоже що пристрій $alias відсутній, ініціалізацію ${DEVICE} відкладено.
Unable to mark loop device as autoclear ||| Не вдалося позначити пристрій петлі (loop) прапорцем autoclear
%s: not configured to support 32-bit little-endian object ||| %s: не налаштовано на підтримку 32-бітових об'єктів з прямим порядком байтів
Memory Bandwidth value exceeding 100 is invalid. ||| Значення ширини каналу пам'яті, що перевищує 100, є некоректним.
array of weight must not contain nulls ||| масив значимості не повинен містити null
Failed to open file %s in read/write mode. ||| Не вдалося відкрити файл %s у режимі читання-запису.
%s must not be called in a subtransaction ||| %s не має викликатися всередині підтранзакції
this remote helper should implement refspec capability ||| цей віддалений помічник має реалізовувати здібність refspec
Copy selection to clipboard ||| Скопіювати позначені об'єкти у буфер обміну
fetch or set the currently defined set of logging filters on daemon ||| отримати або встановити поточний визначений набір фільтрів журналу фонової служби
Error: Your local changes to the following files would be overwritten by merge ||| Помилка: Ваші локальні зміни в наступних файлах були б перезаписані під час злиття
conflicting named address spaces (generic vs %s) for %q+D ||| конфліктуючі іменовані адресні простори (загальні проти %s) для %q+D
deprecated conversion from string constant to %qT ||| застаріла конвертація з рядкової константи в %qT
unrecognized command-line option %<-%s%>; did you mean %<-%s%>? ||| невідома опція командного рядка %<-%s%>; мали на увазі %<-%s%>?
--recurse-submodules can only be used to create branches ||| --recurse-submodules можна використовувати лише для створення гілок
Error while determining whether %s is mounted. ||| Помилка під час спроби визначення, чи змонтовано %s.
bad value for vsetvli immediate field, value must be 0..2047 ||| помилкове значення для поля пришвидшеного використання vsetvli, значення має належати до діапазону 0...2047
List information on all D language transitions. ||| Вивести інформацію про всі переходи мови D.
Move the cursor to a specific line of the window ||| Пересунути курсор до вказаного рядка вікна
Please check whether the specified URI is accessible. ||| Будь ласка, перевірте чи доступна вказана адреса URI.
-mbranch-cost=COST	Set the cost of branches to roughly COST instructions. ||| -mbranch-cost=COST	Встановити вартість гілок приблизно COST інструкцій.
Reduce stack alignment on call sites if possible. ||| Зменшити вирівнювання стеку на місцях виклику, якщо це можливо.
support files larger than 4GB ||| підтримка файлів, розмір яких перевищує 4 ГБ
Action on title bar right-click ||| Дія при клацанні правою кнопкою на заголовку вікна
Error in reading image DIB. ||| Помилка під час читання картинки DIB.
Search backward for a string or a regular expression ||| Шукати рядок або формальний вираз у напрямку до початку тексту
symbol table name section has wrong type: %u ||| розділ назви таблиці символів належить до помилкового типу: %u
descend at most <n> levels ||| спускатися не більше ніж на <н> рівнів
Entry Dir   Time      Size      Name ||| Запис Кат   Час       Розмір    Назва
internal inconsistency: remaining %lu != max %lu; please report this bug ||| внутрішня неузгодженість: залишилося %lu != макс. %lu; будь ласка, повідомте про цю ваду
Control how new windows get focus ||| Спосіб передачі фокусу новим вікнам
'%s' is undocumented; use 'last%s' instead ||| «%s» є недокументованим; скористайтеся замість нього «last%s»
in a call to function %qD declared with attribute %qs ||| у виклику функції %qD, оголошеної з атрибутом %qs
Whether the mouse pointer is visible on the main stage ||| Визначає, чи має бути показано вказівник миші на основній сцені
Explicit shaped array with nonconstant bounds at %C ||| Масив з явною формою з неконстантними межами в %C
Haven't found '%s' in services cache! ||| Не вдалося знайти «%s» у кеші служб!
Opacity of the overlay in outline overlay view mode ||| Непрозорість накладки у режимі перегляду із накладанням ескіза
The following options are language-independent ||| Наступні параметри не залежать від мови
Parent of the current accessible as returned by atk_object_get_parent() ||| Батьківський елемент поточного доступного елемента, який повертатиме atk_object_get_parent()
Ignore first and last points ||| Ігнорувати першу й останню точки
Whether the hash, plus, and asterisk symbols should be visible ||| Визначає, чи має бути показано символи решітки, плюса та зірочки
The named subset to use for this remote ||| Іменований піднабір, яким слід скористатися для цього віддаленого сховища
ISO C++ forbids the use of %qE on explicit instantiations ||| ISO C++ забороняє використання %qE при явних інстанціаціях
size (%ld) out of range, ignored ||| розмір (%ld) лежить поза межами припустимого діапазону, проігноровано
%s tag at %L must be a character string of default kind ||| Мітка %s на %L повинна бути символьним рядком типу за замовчуванням
Compare FILE with local file LOCAL. ||| Порівняти ФАЙЛ з ЛОКАЛЬНИМ_ФАЙЛОМ.
get number of currently active vcpus ||| отримати кількість поточних активних віртуальних процесорів
list aging parameters for the user ||| список параметрів, які застарівають, для користувача
unexpected timeline ID %u in log segment %s, offset %u ||| неочіукваний ID лінії часу %u в сегменті журналу %s, зсув %u
PROCEDURE attribute conflicts with RESULT attribute in %qs at %L ||| Атрибут PROCEDURE конфліктує з атрибутом RESULT в %qs на %L
Align some doubles on dword boundary. ||| Вирівняйте деякі числа з плаваючою комою на границі dword.
Iterator ID exceeds maximum ID of %1$u ||| Ідентифікатор ітератора перевищує максимальне значення ідентифікатора %1$u
Core libraries or services have been updated since boot-up: ||| Основні бібліотеки та служби було оновлено з часу завантаження:
Attempt to put an undefined symbol into set %s ||| Спроба розмістити невідомий символ у множині %s
operand %d out of range ||| операнд %d перебуває поза межами припустимого діапазону
set the default tracking branch to master ||| встановити гілку відстежування за замовчуванням на master
Support RAOINT built-in functions and code generation. ||| Підтримка вбудованих функцій та генерація коду для RAOINT.
Splattered cast metal, with golden highlights ||| Розкидані шматочки металу з золотим відблиском
%qs clause used lexically after first target construct or offloading API ||| %qs використовується лексично після першої конструкції цілі або API віддаленого виконання
<b>Ctrl</b>: make circle or integer-ratio ellipse, snap arc/segment angle ||| <b>Ctrl</b>: створює коло або еліпс з цілим відношенням сторін, обмежує кут дуги/сегмента
Generate source code used to link in the resource file into your code ||| Генерувати початковий код, який використовується для зв'язку з файлом ресурсів вашого коду
volume usage specified, but volume path is missing ||| вказано призначення тому, але не вказано його адреси
explicit instantiation of non-class template %qD ||| явна інстанціація не-класового шаблону %qD
passing %qT to argument %d of %qE, which expects a scalar integer ||| передача %qT в аргумент %d %qE, який очікує скалярний цілочисельне
Wake-on-WLAN mode 'default' and 'ignore' are exclusive flags ||| Прапорці режиму Wake-on-WLAN «default» і «ignore» не можна використовувати одночасно
use ROOT as top level directory ||| використовувати КОРІНЬ як каталог найвищого рівня
Delay focus changes until the pointer stops moving ||| Затримувати зміни фокусу, поки вказівник не перестане рухатись
macro expands to multiple statements ||| макрос розширюється до кількох операторів
Syntax error in CASE specification at %C ||| Синтаксична помилка в специфікації CASE в %C
Mouse button to activate the “Back” command in browser window ||| Кнопка миші, що здійснює команду «Назад» у вікні переглядача
concept %q#D with non-%<bool%> return type %qT ||| концепція %q#D з не-%<bool%> типом повернення %qT
While checking for on-line resizing support ||| Під час перевірки можливості інтерактивної зміни розмірів
Report on permanent memory allocation. ||| Звіт про постійне виділення памʼяті.
Multiple domains exist with the name '%1$s': repeat the request using a UUID ||| Існує декілька доменів із назвою «%1$s»: повторіть запит із використанням UUID
Report bugs to <%s> ||| Адреса для повідомлень про вади: <%s>
grouped targets must provide a recipe ||| для згрупованих цілей має бути надано спосіб збирання
WAL ends before end of online backup ||| WAL завершився до завершення онлайн резервного копіювання
"""


def load_corpus(min_len=30):
    """Повертає трійки (програма, англійський оригінал, український переклад).

    Короткі рядки відкидаємо: «Гаразд» чи «%s» — це підписи кнопок,
    а не речення, і вони перекосили б усю статистику.
    """
    docs = []
    for path in sorted(glob.glob('/usr/share/locale/uk/LC_MESSAGES/*.mo')):
        try:
            with open(path, 'rb') as f:
                catalog = gettext.GNUTranslations(f)
        except Exception:
            continue                     # пошкоджений або чужого формату файл просто пропускаємо
        program = path.split('/')[-1][:-3]
        for source, target in catalog._catalog.items():
            if (isinstance(source, str) and isinstance(target, str)
                    and len(target) > min_len and 'Project-Id' not in target):
                docs.append((program, source, target))
    return docs


docs = load_corpus()
CORPUS_SOURCE = 'локаль системи'

if len(docs) < 2000:                     # локалі немає або вона майже порожня
    CORPUS_SOURCE = 'вбудований мінікорпус'
    docs = []
    for line in MINI_CORPUS.strip().split('\n'):
        source, target = line.split('|||')
        docs.append(('mini', source.strip(), target.strip()))

print("джерело корпусу:", CORPUS_SOURCE)
print("документів:     ", len(docs))
print("програм:        ", len(set(d[0] for d in docs)))
print()
print("приклад запису:")
print("  оригінал :", docs[0][1][:80])
print("  переклад :", docs[0][2][:80])

## 2 · Токенізатор і сходинка «апостроф»Токенізатор той самий, що в темі 02: усе в нижній регістр, слово — це послідовністьукраїнських літер. Одна дрібниця, яка виявиться не дрібницею: **апостроф**.В українських текстах його пишуть щонайменше трьома різними символами, і токенізатор,який знає лише один із них, розрубує слово навпіл.

In [ ]:
# літери української абетки плюс апостроф ʼ (U+02BC) — він усередині слова: зʼєднання
UA_WORD = re.compile(r"[абвгґдежзиійклмнопрстуфхцчшщьюяєіїʼ]+")
# три написання апострофа, які реально трапляються в текстах
APOSTROPHES = re.compile(r"['’`´ʹ‘]")


def ua_tokens(text, fix_apostrophe=True):
    """Ріже український рядок на слова.

    fix_apostrophe=False лишає текст як є — саме так поводиться токенізатор,
    який про різні написання апострофа не знає. Нижче побачимо різницю.
    """
    text = text.lower()
    if fix_apostrophe:
        text = APOSTROPHES.sub('ʼ', text)
    out = []
    for word in UA_WORD.findall(text):
        word = word.strip('ʼ')           # апостроф на краю слова — це не слово
        if word:
            out.append(word)
    return out


example = "Не вдалося зберегти файл: недостатньо пам'яті"
print("сирий рядок      :", example)
print("без нормалізації :", ua_tokens(example, fix_apostrophe=False))
print("з нормалізацією  :", ua_tokens(example, fix_apostrophe=True))

### Скільки коштує кожна сходинкаРахуємо розмір словника після кожного кроку окремо.

In [ ]:
# сирий словник: без нижнього регістру взагалі
RAW_WORD = re.compile(r"[абвгґдежзиійклмнопрстуфхцчшщьюяєіїʼАБВГҐДЕЖЗИІЙКЛМНОПРСТУФХЦЧШЩЬЮЯЄІЇ]+")

vocab_raw   = Counter()       # як є
vocab_lower = Counter()       # нижній регістр
vocab_apo   = Counter()       # нижній регістр + єдиний апостроф

for _, _, target in docs:
    vocab_raw.update(RAW_WORD.findall(target))
    vocab_lower.update(ua_tokens(target, fix_apostrophe=False))
    vocab_apo.update(ua_tokens(target, fix_apostrophe=True))

occurrences = sum(vocab_apo.values())

print("як є              :", f"{len(vocab_raw):>6}", "рядків")
print("нижній регістр    :", f"{len(vocab_lower):>6}", "рядків",
      f"({100*(len(vocab_lower)-len(vocab_raw))/len(vocab_raw):+.1f} %)")
print("єдиний апостроф   :", f"{len(vocab_apo):>6}", "рядків",
      f"({len(vocab_apo)-len(vocab_lower):+d} до попереднього)")
print()
print("слововживань після нормалізації:", occurrences,
      f"({sum(vocab_lower.values())-occurrences} було розрубаними половинками)")

# що саме зникло й що зʼявилось
gone = set(vocab_lower) - set(vocab_apo)
came = set(vocab_apo) - set(vocab_lower)
print()
print("уламків зникло   :", len(gone), sorted(gone, key=lambda w: -vocab_lower[w])[:6])
print("слів зʼявилось   :", len(came), sorted(came, key=lambda w: -vocab_apo[w])[:6])

## 3 · Лематизація: лема й граматичні ознаки`pymorphy3` з `lang='uk'` повертає **список розборів**. У кожного розбору є `normal_form`(лема) і `tag` (граматичні ознаки). Друге поле лематизація викидає — і саме через цедалі почнуться і користь, і шкода.

In [ ]:
morph = pymorphy3.MorphAnalyzer(lang='uk')

for parse in morph.parse('зламаного')[:3]:
    print(f"{parse.normal_form:<12} {parse.tag}")

print()
print("перший розбір «зламаного»:")
print("  лема   :", morph.parse('зламаного')[0].normal_form)
print("  ознаки :", morph.parse('зламаного')[0].tag)

### Лематизуємо весь словникРозбираємо **різні словоформи**, а не слововживання: результат для слова завжди той самий, тож рахувати його двічі немає сенсу.

In [ ]:
forms = sorted(vocab_apo)                       # усі різні словоформи корпусу

start = time.time()
lemma_of = {}
for form in forms:
    lemma_of[form] = morph.parse(form)[0].normal_form
lemmatize_seconds = time.time() - start

lemmas = set(lemma_of.values())
drop = 100 * (1 - len(lemmas) / len(forms))

print("словоформ :", len(forms))
print("лем       :", len(lemmas))
print(f"падіння   : {drop:.1f} %")
print(f"час       : {lemmatize_seconds:.2f} с на {len(forms)} словоформ")

assert len(lemmas) < len(forms), "лематизація зобовʼязана зменшувати словник"

### Розсип форм: скільки рядків дає одне словоСереднє тут майже нічого не описує — подивимось на розподіл цілком.

In [ ]:
forms_of_lemma = defaultdict(list)
for form, lemma in lemma_of.items():
    forms_of_lemma[lemma].append(form)

sizes = [len(v) for v in forms_of_lemma.values()]
single = sum(1 for s in sizes if s == 1)
biggest = max(sizes)
champions = [lemma for lemma, v in forms_of_lemma.items() if len(v) == biggest]

print(f"форм на лему в середньому : {len(forms)/len(lemmas):.2f}")
print(f"лем рівно з однією формою : {single} ({100*single/len(lemmas):.1f} %)")
print(f"найбільше форм в однієї леми: {biggest}")
print(f"лем із таким рекордом      : {len(champions)}")

# хто саме тримає рекорд — частини мови рекордсменів
pos_counts = Counter(str(morph.parse(lemma)[0].tag.POS) for lemma in champions)
print("частини мови рекордсменів  :", pos_counts.most_common(4))

example_lemma = 'поточний' if 'поточний' in forms_of_lemma else champions[0]
print()
print(f"усі форми леми «{example_lemma}»:")
print("  " + " · ".join(sorted(forms_of_lemma[example_lemma],
                               key=lambda w: -vocab_apo[w])))

## 4 · Наївний стемінг: дві ціни одночасноНайпростіший стемер, який лише можна уявити: **відріж останні `k` літер**. Єдинийзапобіжник — не вкорочувати слово менше ніж до `min_stem` літер.Питань до нього два, і дивитись на них треба **разом**:1. скільки різних основ вийшло — це користь;2. скільки основ склеїли словоформи з **різними лемами** — це шкода.

In [ ]:
def naive_stem(word, k, min_stem=3):
    """Відрізає k літер із кінця, але не робить основу коротшою за min_stem."""
    if len(word) - k >= min_stem:
        return word[:len(word) - k]
    return word                          # коротке слово лишаємо цілим


def measure_stemmer(k, min_stem):
    """Повертає (скільки основ, скільки основ склеїли різні леми)."""
    lemmas_under_stem = defaultdict(set)
    for form in forms:
        lemmas_under_stem[naive_stem(form, k, min_stem)].add(lemma_of[form])
    glued = sum(1 for lemma_set in lemmas_under_stem.values() if len(lemma_set) > 1)
    return len(lemmas_under_stem), glued


print(f"словоформ: {len(forms)},  лем після розбору: {len(lemmas)}")
print()
print("мін. основа   k   основ   стиснення   склеєно основ")
for min_stem in (2, 3, 4):
    for k in (1, 2, 3, 4, 5):
        n_stems, glued = measure_stemmer(k, min_stem)
        print(f"{min_stem:>10}  {k:>3}  {n_stems:>6}   "
              f"{100*(1-n_stems/len(forms)):>7.1f} %   "
              f"{glued:>5}  ({100*glued/n_stems:>4.1f} %)")
    print()

### Жодне налаштування не дістає до лематизаціїПеребираємо всю сітку й шукаємо найкраще стиснення, яке взагалі дає обрізання.

In [ ]:
best = None
for min_stem in (2, 3, 4):
    for k in (1, 2, 3, 4, 5):
        n_stems, glued = measure_stemmer(k, min_stem)
        if best is None or n_stems < best[0]:
            best = (n_stems, glued, k, min_stem)

n_stems, glued, k, min_stem = best
print(f"найменший словник у всій сітці: {n_stems} основ")
print(f"  досягнуто при k={k}, мінімальна основа {min_stem}")
print(f"  ціна: {glued} склеєних основ ({100*glued/n_stems:.1f} %)")
print(f"лематизація дає {len(lemmas)} лем і нуль склеєних")
print(f"різниця: обрізання лишає на {n_stems - len(lemmas)} рядків більше")

assert n_stems > len(lemmas), "обрізання несподівано обігнало лематизацію — перевір код"

### Що саме склеюєтьсяНайцікавіші випадки — короткі слова, де рахунок літер із кінця доходить до кореня.

In [ ]:
groups = defaultdict(list)
for form in forms:
    groups[naive_stem(form, 3, 3)].append(form)

# сортуємо за сумарною частотою: спершу ті склейки, які реально псують текст
broken = [(stem, group) for stem, group in groups.items()
          if len({lemma_of[w] for w in group}) > 1]
broken.sort(key=lambda pair: -sum(vocab_apo[w] for w in pair[1]))

for stem, group in broken[:8]:
    top_forms = sorted(group, key=lambda w: -vocab_apo[w])[:5]
    lemmas_here = sorted({lemma_of[w] for w in group})[:5]
    print(f"основа «{stem}»: {', '.join(top_forms)}")
    print(f"    леми: {', '.join(lemmas_here)}")

### Слова з чергуванням основиТі самі приклади, що в другому інтерактиві лекції: обрізання проти розбору.

In [ ]:
word_groups = [
    ['нога', 'нозі', 'ніг', 'ногами'],
    ['рука', 'руці', 'рук', 'руками'],
    ['книжка', 'книжці', 'книжок', 'книжками'],
    ['друг', 'друзі', 'друга', 'другом'],
    ['ходити', 'ходжу', 'ходить', 'ходив'],
]

for group in word_groups:
    lemmas_here = [morph.parse(w)[0].normal_form for w in group]
    print(f"{group[0]:<8} леми: {len(set(lemmas_here))} різних → {lemmas_here}")
    for k in (1, 2, 3, 4, 5):
        stems_here = [naive_stem(w, k) for w in group]
        print(f"          k={k}: {len(set(stems_here))} різних → {stems_here}")
    print()

## 5 · Омонімія: перший розбір — просто перший`morph.parse(word)[0]` бере **перший** розбір. Природно вважати, що перший —найімовірніший. Перевіримо це числом.

In [ ]:
all_parses = {form: morph.parse(form) for form in forms}

multi = [f for f in forms if len(all_parses[f]) > 1]
ambiguous = [f for f in forms if len({p.normal_form for p in all_parses[f]}) > 1]

def share_of_text(word_list):
    return 100 * sum(vocab_apo[w] for w in word_list) / occurrences

print(f"словоформ усього              : {len(forms)}")
print(f"з більш ніж одним розбором    : {len(multi)} "
      f"({100*len(multi)/len(forms):.1f} % словника, {share_of_text(multi):.1f} % тексту)")
print(f"з РІЗНИМИ лемами в розборах   : {len(ambiguous)} "
      f"({100*len(ambiguous)/len(forms):.1f} % словника, {share_of_text(ambiguous):.1f} % тексту)")

### Чи допомагає `score`У кожного розбору є оцінка правдоподібності. Подивимось, чи вона взагалі щось розрізняє.

In [ ]:
gaps = [all_parses[f][0].score - all_parses[f][1].score for f in ambiguous]
close = sum(1 for gap in gaps if gap < 0.05)

print(f"медіанна різниця оцінок двох найкращих розборів: {np.median(gaps):.4f}")
print(f"різниця менша за 0.05: {close} із {len(ambiguous)} "
      f"({100*close/len(ambiguous):.1f} %)")

### Автоматичний детектор хибного виборуЯкщо обрана лема не трапляється в корпусі **жодного разу**, а якась із наступних трапляється, вибір майже напевно хибний. Це можна порахувати без жодної ручної розмітки.

In [ ]:
suspicious = []
for form in ambiguous:
    chosen = all_parses[form][0].normal_form
    others = [p.normal_form for p in all_parses[form][1:]]
    if chosen not in vocab_apo and any(other in vocab_apo for other in others):
        suspicious.append(form)

print(f"перша лема не трапляється в корпусі, а альтернативна трапляється:")
print(f"  {len(suspicious)} із {len(ambiguous)} неоднозначних "
      f"({100*len(suspicious)/len(ambiguous):.1f} %)")
print()
for form in sorted(suspicious, key=lambda w: -vocab_apo[w])[:10]:
    better = [p.normal_form for p in all_parses[form] if p.normal_form in vocab_apo]
    print(f"  {form:<10} {vocab_apo[form]:>5} разів → «{lemma_of[form]}»"
          f"   краще: {better[:2]}")

### Вісім слів із лекціїТі самі, що в четвертому інтерактиві. Дивимось на всі розбори кожного.

In [ ]:
for word in ['бути', 'дані', 'його', 'роботу', 'меню', 'домен', 'при', 'має']:
    parses = morph.parse(word)
    print(f"{word}  ({vocab_apo.get(word, 0)} разів у корпусі)")
    seen = set()
    for parse in parses[:4]:
        key = (parse.normal_form, str(parse.tag))
        if key in seen:
            continue
        seen.add(key)
        in_corpus = vocab_apo.get(parse.normal_form, 0)
        mark = '←' if parse is parses[0] else ' '
        print(f"   {mark} {parse.normal_form:<10} {str(parse.tag):<32} "
              f"score={parse.score:.2f}  лема в корпусі: {in_corpus}")
    print()

## 6 · Задача, на якій усе перевірятимемоКорпус **паралельний**: поруч із українським рядком лежить англійський оригінал.Це дозволяє зробити чесний дослід — **мітку взяти з англійської, а ознаки з української**.Класифікатор англійського тексту не бачить узагалі, тож підглянути відповідь не може.Мітка: чи є в англійському оригіналі слово про невдачу або заперечення.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GroupShuffleSplit

NEGATION = re.compile(
    r"\b(cannot|can't|could not|couldn't|failed|unable|not|no|never|without|invalid)\b",
    re.IGNORECASE)

# 20 тисяч документів досить: на всьому корпусі числа ті самі, а чекати втричі довше
sample = list(docs)
random.Random(0).shuffle(sample)
sample = sample[:20000]

texts_forms, texts_lemmas, labels = [], [], []
for _, source, target in sample:
    tokens = ua_tokens(target)
    if len(tokens) < 3:                  # надто короткі рядки нічого не вирішують
        continue
    texts_forms.append(' '.join(tokens))
    texts_lemmas.append(' '.join(lemma_of.get(t, morph.parse(t)[0].normal_form) for t in tokens))
    labels.append(1 if NEGATION.search(source) else 0)

labels = np.array(labels)
print("прикладів      :", len(labels))
print(f"частка позитивних: {labels.mean():.4f}")
# скільки дало б тупе «завжди відповідай більшим класом» — з цим і порівнюємо
print(f"вгадування навмання: {max(labels.mean(), 1 - labels.mean()):.4f}")
print()
print("той самий рядок у двох поданнях:")
print("  словоформи:", texts_forms[0][:80])
print("  леми      :", texts_lemmas[0][:80])

### Перевірка: мішок слів усередині `CountVectorizer` — не магіяПерш ніж на нього спиратися, переконаймось, що він рахує рівно те саме, що й наш `Counter`.

In [ ]:
one_document = texts_forms[0]
vectorizer = CountVectorizer(token_pattern=r'\S+')
matrix = vectorizer.fit_transform([one_document])

library_counts = dict(zip(vectorizer.get_feature_names_out(), matrix.toarray()[0]))
our_counts = dict(Counter(one_document.split()))

assert library_counts == our_counts, "розрахунок розійшовся!"
print("✅ збігається:", sorted(our_counts.items())[:6])

## 7 · Коли лематизація допомагаєОдин і той самий дослід на різному обсязі навчальної вибірки — і **на трьох зернах**,бо різниця, менша за розкид, не є різницею.

In [ ]:
def accuracy_three_seeds(texts, train_size, test_size=4000):
    """Точність на трьох різних поділах. Повертає (середнє, найгірше, найкраще, ознак)."""
    scores, n_features = [], 0
    for seed in (0, 1, 2):
        train_x, test_x, train_y, test_y = train_test_split(
            texts, labels, test_size=test_size, train_size=train_size,
            random_state=seed, stratify=labels)
        # min_df=2 викидає слова, що трапились у навчанні один раз: користі з них нема
        vec = CountVectorizer(token_pattern=r'\S+', min_df=2)
        train_matrix = vec.fit_transform(train_x)
        test_matrix = vec.transform(test_x)
        n_features = train_matrix.shape[1]
        model = LogisticRegression(max_iter=400).fit(train_matrix, train_y)
        scores.append(model.score(test_matrix, test_y))
    return float(np.mean(scores)), min(scores), max(scores), n_features


print("навчання   словоформи (розкид)            леми (розкид)                різниця")
for train_size in (200, 1000, 3000, 10000):
    f_mean, f_lo, f_hi, f_n = accuracy_three_seeds(texts_forms, train_size)
    l_mean, l_lo, l_hi, l_n = accuracy_three_seeds(texts_lemmas, train_size)
    overlap = (l_lo <= f_hi and f_lo <= l_hi)
    print(f"{train_size:>8}   {f_mean:.4f} ({f_lo:.4f}-{f_hi:.4f}) {f_n:>5} озн."
          f"   {l_mean:.4f} ({l_lo:.4f}-{l_hi:.4f}) {l_n:>5} озн."
          f"   {l_mean-f_mean:+.4f}"
          f"   {'розкид більший' if overlap else 'різниця справжня'}")

## 8 · Коли лематизація шкодитьТепер задача, у якій відповідь лежить **саме в закінченні**. Розмітку не вигадуємо:файли перекладів зберігають **форми множини**, і поставили їх люди-перекладачі.Українська має три форми — беремо першу («одна річ») і третю («багато речей»).

In [ ]:
MINI_PLURAL = """
запущений %d паралельний виконавець очистки для очищення індексу (заплановано: %d) ||| запущено %d паралельних виконавців очистки для очищення індексу (заплановано: %d)
За %d секунду не знайдено жодного додатка; скасування… ||| За %d секунд не знайдено жодного додатка; скасування…
велике число (bignum) обрізано до %d байта ||| велике число (bignum) обрізано до %d байтів
Наступна %d латка вимагає перезавантаження системи: ||| Наступні %d латок вимагають перезавантаження системи:
Вибрано %'d інший об'єкт ||| Вибрано %'d інших об'єктів
%d помилка імпортування ||| %d помилок імпортування
не вдалося записати %llu елемент до %s: %s ||| не вдалося записати %llu елементів до %s: %s
Не вдалося встановити %d пакунок: ||| Не вдалося встановити %d пакунків:
пізніше оголошено як %s з %u неспецифікованою звʼязаною змінною ||| пізніше оголошено як %s з %u невизначеними звʼязаними змінними
Буде перевстановлено наступний %d продукт: ||| Буде перевстановлено наступних %d продуктів:
Оголошення віртуального деструктора %qD як final дозволить девіртуалізацію %i виклику, виконаного %lli разів ||| Оголошення віртуального деструктора %qD як final дозволить девіртуалізацію %i викликів, виконаних %lli разів
Введіть або змініть текст (%d символ%s); <b>Enter</b> — початок нового рядка. ||| Введіть або змініть текст (%d символів%s); <b>Enter</b> — початок нового рядка.
Наступний %d пакунок рекомендовано, але не буде встановлено через конфлікти або проблеми з залежностями: ||| Наступних %d пакунків рекомендовано, але не буде встановлено через конфлікти або проблеми з залежностями:
Неможливо переміститися на наступний %d байт за допомогою seek(). ||| Неможливо переміститися на наступні %d байтів за допомогою seek().
Локальне посилання налаштовано для 'git push'%s: ||| Локальних посилань налаштовано для 'git push'%s:
Вилучити %d дубльований об'єкт ||| Вилучити %d дубльованих об'єктів
Імпортовано %u контакт з «%s». ||| Імпортовано %u контактів з «%s».
Фільтруємо нові повідомлення у «%s : %s» ||| Фільтруємо нові повідомлення у «%s : %s»
Схвалена мікропрограма: ||| Схвалені мікропрограми:
Вилучено <b>%i</b> непотрібний елемент у &lt;defs&gt;. ||| Вилучено <b>%i</b> непотрібних елементів у &lt;defs&gt;.
параметр «-%s» проігноровано ||| параметри «-%s» проігноровано
За %1$u дня та %2$u хвилину ||| За %1$u дня та %2$u хвилин
але деякі повідомлення мають %lu множинну форму ||| але деякі повідомлення мають %lu множинних форм
неправильний розмір файлу '%s': %lld байт ||| неправильний розмір файлу '%s': %lld байтів
Директива %<%.*s%> записує %wu байт у регіон розміром між %wu і %wu ||| Директива %<%.*s%> записує %wu байтів у регіон розміром між %wu і %wu
Відповідника параметра не знайдено: {} ||| Відповідника параметрів не знайдено: {}
Ділянку заповнено, створено контур з <b>%d</b> вузлом. ||| Область заповнено, створено контур з <b>%d</b> вузлами.
вернути до старішої версії ||| вернути до старіших версій
Відсутній заголовок підпису у повідомленні, але тіло повідомлення займає %u байт ||| Відсутній заголовок підпису у повідомленні, але тіло повідомлення займає %u байтів
Диск справний, %d атрибут помилковий ||| Диск справний, %d атрибутів помилкові
Наступне %d оновлення програм НЕ буде встановлено: ||| Наступні %d оновлень програм НЕ буде встановлено:
%d типове клавіатурне скорочення знайдено. ||| %d типових клавіатурних скорочень знайдено.
Буде перевстановлено наступну %d програму: ||| Буде перевстановлено наступних %d програм:
Вибрано один вус, що містить %d опорну точку (перетягніть з клавішею <b>Shift</b>, щоб роз'єднати) ||| Вибрано один вус, що містить %d опорних точок (перетягніть з клавішею <b>Shift</b>, щоб роз'єднати)
%u тип файлів і посилань, які може відкривати програма ||| %u типів файлів і посилань, які може відкривати програма
По_вернути дублювання %d об'єкта ||| По_вернути дублювання %d об'єктів
Буде перевстановлено наступний %d шаблон: ||| Буде перевстановлено наступних %d шаблонів:
'%s' сховище було додане до вимкнених сховищ сервісу '%s' ||| '%s' сховищ були додані до вимкнених сховищ сервісу '%s'
Призупинити обробку на {} секунду ||| Призупинити обробку на {} секунд
Перебазування %s на %s (%d команда) ||| Перебазування %s на %s (%d команд)
всього помилок перевірки контрольних сум: %lld ||| всього помилок перевірки контрольних сум: %lld
Вилучено %lu блокування. ||| Вилучено %lu блокувань.
Знайдено <b>%d</b> об'єкт (з <b>%d</b>), %s відповідність. ||| Знайдено <b>%d</b> об'єктів (з <b>%d</b>), %s відповідність.
Там буде %1% збіг для '%2%'. ||| Там буде %1% збігів для '%2%'.
%qD доступ до %wu байт за зміщеннями %s і %s може перекриватися з до %wu байтів за зміщенням %s ||| %qD доступ до %wu байтів за зміщеннями %s і %s може перекриватися з до %wu байтів за зміщенням %s
знайдено незавершений tar-заголовок (%lu байт) ||| знайдено незавершений tar-заголовок (%lu байт)
Остаточно вилучити %'d вибраний пункт? ||| Остаточно вилучити %'d вибраних пунктів?
%u 2-фазовий стан файлу був записаний завдяки довготривалій підготовленій транзакції ||| %u 2-фазовий стан файлів був записаний завдяки довготривалим підготовленим транзакціям
Доступні програми для %s ||| Доступні програми для %s
Інструкція не вкладається у доступні слоти затримки (інструкція у %d слів, лишився %d слот) ||| Інструкція не вкладається у доступні слоти затримки (інструкція у %d слів, лишилося %d слотів)
Буде перевстановлено наступний %d шаблон: ||| Буде перевстановлено наступні %d шаблонів:
Групове перейменування %d файлу ||| Групове перейменування %d файлів
Сервер Google зайнято, очікуємо на повторну спробу (%d:%02d хвилина) ||| Сервер Google зайнято, очікуємо на повторну спробу (%d:%02d хвилин)
За %1$u години та %2$u хвилину ||| За %1$u години та %2$u хвилин
Не вдалося знайти %s, потрібної для %s. Будь ласка, ознайомтеся із %s, щоб дізнатися більше. ||| Не вдалося знайти %s, потрібної для %s. Будь ласка, ознайомтеся із %s, щоб дізнатися більше.
Була запропонована наступна %d латка, але її не буде встановлено: ||| Було запропоновано наступних %d латок, але вони не будуть встановлені:
%'d / %'d — залишилось %s ||| %'d / %'d — залишилось %s
Стиснено %'d файл в «%s» ||| Стиснено %'d файлів в «%s»
%qD доступ до %wu до %wu байтів за зміщеннями %s і %s перекривається з %wu байтом за зміщенням %s ||| %qD доступ до %wu до %wu байтів за зміщеннями %s і %s перекривається з %wu байтами за зміщенням %s
%qD вказує на більш обмежувальний атрибут, ніж його ціль %qD: %s ||| %qD вказує на більш обмежувальні атрибути, ніж його ціль %qD: %s
Проблеми із залежностями модуля з Defaults: ||| Проблеми із залежностями модулів з Defaults:
знайдено %d операнд «%s»: мало бути %d ||| знайдено %d операндів «%s»: мало бути %d
Пакунок містить це %llu посилання: ||| Пакунок містить ці %llu посилань:
Замінено %1 відповідник ||| Замінено %1 відповідників
база даних '%s' повинна бути очищена (vacuumed), перед тим як більшість MultiXactId буде використана (%u) ||| баз даних '%s' повинні бути очищені (vacuumed) перед тим, як більшість MultiXactIds буде використано (%u)
тільки %u імʼя надано для структурованого привʼязування ||| тільки %u імен надано для структурованого привʼязування
Розмір сегменту WAL повинен задаватись ступенем 2 в інтервалі від 1 МБ до 1 ГБ, але в керуючому файлі вказано значення %d ||| Розмір сегменту WAL повинен задаватись ступенем 2 в інтервалі від 1 МБ до 1 ГБ, але в керуючому файлі вказано значення %d
Скасувати позначення зірками %d файлу ||| Скасувати позначення зірками %d файлів
%uпристрій не є найкращою відомою конфігурацією. ||| %u пристрої не є найкращою відомою конфігурацією.
Вказане блокування було успішно здійснено. ||| Вказані блокування було успішно здійснено.
"""

def load_plural_pairs():
    """Пари «одна річ / багато речей» з файлів перекладів.

    Ключ у каталозі для множини — кортеж (текст, номер форми). Форма 0 — «одна»,
    форма 2 — «багато». Беремо лише ті повідомлення, де перекладач заповнив обидві.
    """
    by_message = defaultdict(dict)
    for path in sorted(glob.glob('/usr/share/locale/uk/LC_MESSAGES/*.mo')):
        try:
            with open(path, 'rb') as f:
                catalog = gettext.GNUTranslations(f)
        except Exception:
            continue
        for key, translated in catalog._catalog.items():
            if (isinstance(key, tuple) and isinstance(translated, str)
                    and key[1] in (0, 2) and len(translated) > 10):
                by_message[(path, key[0])][key[1]] = translated
    return [(v[0], v[2]) for v in by_message.values() if 0 in v and 2 in v]


plural_pairs = load_plural_pairs()
PLURAL_SOURCE = 'локаль системи'
if len(plural_pairs) < 100:
    PLURAL_SOURCE = 'вбудований зразок'
    plural_pairs = []
    for line in MINI_PLURAL.strip().split('\n'):
        one, many = line.split('|||')
        plural_pairs.append((one.strip(), many.strip()))

print("джерело пар:", PLURAL_SOURCE)
print("пар «одна / багато»:", len(plural_pairs))
print()
one, many = plural_pairs[0]
print("одна   :", one)
print("багато :", many)
print("леми   :", ' '.join(morph.parse(w)[0].normal_form for w in ua_tokens(one)))
print("леми   :", ' '.join(morph.parse(w)[0].normal_form for w in ua_tokens(many)))

### Знаходимо пару, у якої після лематизації лишається однаковий рядокЦе і є механізм шкоди, показаний найпростіше.

In [ ]:
def to_lemmas(text):
    return ' '.join(lemma_of.get(w, morph.parse(w)[0].normal_form) for w in ua_tokens(text))


collapsed = 0
first_example = None
for one, many in plural_pairs:
    if to_lemmas(one) == to_lemmas(many):
        collapsed += 1
        if first_example is None:
            first_example = (one, many)

print(f"пар, які після лематизації стали ОДНАКОВИМ рядком: "
      f"{collapsed} із {len(plural_pairs)} ({100*collapsed/len(plural_pairs):.1f} %)")
print()
one, many = first_example
print("одна        :", one)
print("багато      :", many)
print("леми (одна) :", to_lemmas(one))
print("леми (багато):", to_lemmas(many))

### Міряємо точністьДілимо **за парами**, щоб два варіанти того самого речення не потрапили по різні боки. Три зерна.

In [ ]:
plural_texts_forms, plural_texts_lemmas, plural_labels, plural_groups = [], [], [], []
for index, (one, many) in enumerate(plural_pairs):
    for text, label in ((one, 0), (many, 1)):
        tokens = ua_tokens(text)
        plural_texts_forms.append(' '.join(tokens))
        plural_texts_lemmas.append(to_lemmas(text))
        plural_labels.append(label)
        plural_groups.append(index)          # обидва варіанти однієї пари — одна група

plural_labels = np.array(plural_labels)
plural_groups = np.array(plural_groups)
coin = max(plural_labels.mean(), 1 - plural_labels.mean())


def plural_accuracy(texts):
    scores, n_features = [], 0
    for seed in (0, 1, 2):
        splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=seed)
        train_index, test_index = next(splitter.split(texts, plural_labels, plural_groups))
        vec = CountVectorizer(token_pattern=r'\S+', min_df=2)
        train_matrix = vec.fit_transform([texts[i] for i in train_index])
        test_matrix = vec.transform([texts[i] for i in test_index])
        n_features = train_matrix.shape[1]
        model = LogisticRegression(max_iter=400).fit(train_matrix, plural_labels[train_index])
        scores.append(model.score(test_matrix, plural_labels[test_index]))
    return float(np.mean(scores)), min(scores), max(scores), n_features


print(f"прикладів: {len(plural_labels)}, груп: {len(set(plural_groups))}")
print(f"вгадування навмання: {coin:.4f}")
print()
scores = {}
for name, texts in (('словоформи', plural_texts_forms), ('леми', plural_texts_lemmas)):
    mean, low, high, n_features = plural_accuracy(texts)
    scores[name] = mean
    print(f"{name:<11} {mean:.4f}  (розкид {low:.4f}-{high:.4f}, {n_features} ознак)")

print()
print(f"що дала лематизація: {scores['леми'] - scores['словоформи']:+.4f}")

## 9 · Стоп-слова: будуємо список саміГотового списку не беремо — він не показує, звідки взявся. Сортуємо словник зачастотою й дивимось на верхівку.

In [ ]:
print("топ-25 слів корпусу:")
for rank, (word, count) in enumerate(vocab_apo.most_common(25), 1):
    print(f"  {rank:>2}. {word:<12} {count:>6}")

ranked = [word for word, _ in vocab_apo.most_common(400)]
print()
print("довжина списку   покрито тексту   покрито словника")
for n in (1, 2, 5, 10, 20, 50, 100, 200):
    covered = sum(vocab_apo[w] for w in ranked[:n])
    print(f"{n:>14}   {100*covered/occurrences:>13.1f} %   "
          f"{100*n/len(forms):>15.2f} %")

# а тепер — скільки з цього виграє САМ словник лем
stop_lemmas = {lemma_of[w] for w in ranked[:20]}
print()
print("лем усього                     :", len(lemmas))
print("лем після викидання 20 стоп-слів:", len(lemmas - stop_lemmas))

### Скільки коштує кожна довжина спискуТа сама задача, що в розділі 7. Три зерна.

In [ ]:
def accuracy_without(stop_words):
    scores, n_features = [], 0
    for seed in (0, 1, 2):
        train_x, test_x, train_y, test_y = train_test_split(
            texts_forms, labels, test_size=0.25, random_state=seed, stratify=labels)
        vec = CountVectorizer(token_pattern=r'\S+', min_df=2,
                              stop_words=stop_words if stop_words else None)
        train_matrix = vec.fit_transform(train_x)
        test_matrix = vec.transform(test_x)
        n_features = train_matrix.shape[1]
        model = LogisticRegression(max_iter=400).fit(train_matrix, train_y)
        scores.append(model.score(test_matrix, test_y))
    return float(np.mean(scores)), min(scores), max(scores), n_features


baseline = None
print("викинуто   точність (розкид)          ознак   втрачено")
for n in (0, 1, 2, 5, 10, 20, 50, 100, 200):
    mean, low, high, n_features = accuracy_without(ranked[:n])
    if baseline is None:
        baseline = mean
    print(f"{n:>8}   {mean:.4f} ({low:.4f}-{high:.4f})   {n_features:>5}   {baseline-mean:+.4f}")

## 10 · Ціна в часі: кеш вирішує всеЛема слова не залежить від контексту, тож розібравши слово раз, результат можназапамʼятати назавжди. Наскільки це важливо — питання не смаку, а числа.

In [ ]:
stream = []
for _, _, target in docs:
    stream.extend(ua_tokens(target))

slice_size = min(100000, len(stream))
piece = stream[:slice_size]
print(f"слововживань у корпусі: {len(stream)}")
print(f"беремо зріз          : {slice_size}")
print(f"різних словоформ у зрізі: {len(set(piece))} "
      f"({100*(1-len(set(piece))/slice_size):.1f} % викликів кеш прибирає)")

start = time.time()
for word in piece:
    morph.parse(word)[0].normal_form
seconds_no_cache = time.time() - start

cache = {}
start = time.time()
for word in piece:
    if word not in cache:
        cache[word] = morph.parse(word)[0].normal_form
    cache[word]
seconds_with_cache = time.time() - start

print()
print(f"без кешу : {seconds_no_cache:.2f} с")
print(f"з кешем  : {seconds_with_cache:.2f} с")
print(f"прискорення: {seconds_no_cache/seconds_with_cache:.1f}×")
print(f"на мільйон слововживань це {seconds_no_cache*1e6/slice_size:.0f} с проти "
      f"{seconds_with_cache*1e6/slice_size:.0f} с")

## Завдання### 🟢 Рівень 1Додай до сітки стемера значення `k = 6` і `k = 7` при мінімальній основі 2.Чи знайдеться налаштування, яке дає менше основ, ніж лематизація? Надрукуй відповідь числом.### 🟡 Рівень 2Візьми задачу про множину з розділу 8 і додай третє подання: **лема плюс граматичнечисло з `tag`** (наприклад `хвилина|sing`). Заміряй точність на трьох зернах.Чи повернувся результат до рівня словоформ?### 🔴 Рівень 3Побудуй розумніший вибір розбору: якщо лема першого розбору не трапляється в корпусі,а лема якогось наступного трапляється — бери наступний. Заміряй, на скількизміниться розмір словника лем і скільки словоформ отримають іншу лему.